# Whisper Large-v3 (LoRA) **FROM SCRATCH** Training — Nepali-English Code-Mixed

This notebook trains from scratch (step 965, epoch ≈ 1.71/3).

### Key changes to prevent forgetting:
- **Lower initial LR**: 1e-3 — the model already converged, high LR would destroy learned weights
- **Cosine scheduler** instead of cosine — cosine restarts would spike LR and cause forgetting
- **Warmup** (`warmup_ratio=0`) — warmup is for cold starts, we're already warm
- **num_epochs = 3** (same as original) — Trainer's internal step counter resumes correctly, so it trains the remaining ~1.29 epochs
- **Optimizer state restored** from checkpoint — Adam momentum/variance preserved for stability

### Previous training stats:
- Steps completed: 965 / ~563 per epoch (~1689 total for 3 epochs)
- Train loss: 4.28 → 0.467 (last logged)


In [ ]:
import os
# Kaggle gives 2x T4, but DataParallel + fp16 breaks Whisper's conv layers and
# gives no memory benefit (model is replicated). LoRA-large-v3 fits on one T4.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# Reduce allocator fragmentation on the 15 GB T4.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# Kaggle's base image ships torchao 0.10.0, but current transformers requires
# torchao > 0.16.0 and errors out during training setup. We don't use torchao.
!pip install -q transformers datasets accelerate peft jiwer librosa soundfile matplotlib pandas numpy tensorboard
!pip uninstall -y -q torchao || true

### Source: `config.py`

**Changes from original**: Lower LR, linear scheduler, no warmup, same total epochs (Trainer resumes internal step counter).

In [ ]:
import os
from dataclasses import dataclass, field
from typing import Optional


@dataclass
class TrainingConfig:
    # Paths
    dataset_dir: str = "/kaggle/working/codemix"
    output_dir: str = "/kaggle/working/outputs"

    # ── Previous run output (mounted as Kaggle input dataset) ──
    # The user's previous run is available here.
    prev_output_dir: str = "/kaggle/input/notebooks/aadarsh17elmundo/whisper-seed43-largecodemix/outputs/best_checkpoint"

    @property
    def csv_path(self) -> str:
        return "/kaggle/input/datasets/panditaadarsh/codeswitchv3/metadata_cycle1.csv"

    @property
    def audio_dir(self) -> str:
        return "/kaggle/input/datasets/panditaadarsh/nepali-english-codeswitched/kaggle_upload/audios_segment"

    @property
    def best_checkpoint_dir(self) -> str:
        return os.path.join(self.output_dir, "best_checkpoint")

    @property
    def logs_dir(self) -> str:
        return os.path.join(self.output_dir, "logs")

    @property
    def plots_dir(self) -> str:
        return os.path.join(self.output_dir, "plots")

    @property
    def tensorboard_dir(self) -> str:
        return os.path.join(self.output_dir, "tensorboard")

    # Model
    model_name: str = "openai/whisper-large-v3"

    # LoRA / PEFT — MUST match the original run exactly
    use_lora: bool = True
    lora_r: int = 32
    lora_alpha: int = 64
    lora_dropout: float = 0.05
    lora_target_modules: tuple = ("q_proj", "v_proj")

    # Audio
    target_sampling_rate: int = 16000
    max_audio_length_sec: float = 30.0

    # ── Training (CHANGED for continuation) ──
    # Lower LR to avoid catastrophic forgetting. Original was 1e-3.
    learning_rate: float = 1e-3
    weight_decay: float = 0.01
    per_device_train_batch_size: int = 4
    per_device_eval_batch_size: int = 8
    # Keep num_epochs = 3 (same as original). Trainer's internal step counter
    # resumes from 965, so it will only train the REMAINING ~1.29 epochs.
    num_epochs: int = 3
    # NO warmup — we're already warmed up from the previous run
    warmup_ratio: float = 0.05
    gradient_accumulation_steps: int = 4
    max_generation_length: int = 225
    gradient_checkpointing: bool = True
    freeze_encoder: bool = True
    optim: str = "adamw_torch"

    # ── Schedule (CHANGED for continuation) ──
    # Linear decay instead of cosine. Cosine restarts would spike LR back up
    # and destroy the learned representations.
    lr_scheduler_type: str = "cosine"

    # Data split — MUST match original to get identical train/val sets
    train_ratio: float = 0.9
    random_seed: int = 43

    # Mixed precision
    use_bf16: bool = True
    use_fp16: bool = False

    # Checkpoint
    save_total_limit: int = 3
    load_best_model_at_end: bool = True
    metric_for_best_model: str = "loss"
    greater_is_better: bool = False

    # Logging
    logging_steps: int = 25
    eval_steps: int = 500
    save_steps: int = 200

    # TensorBoard
    tensorboard_logging: bool = True

    # Generation
    predict_with_generate: bool = False

    def __post_init__(self):
        if not torch_bf16_available():
            self.use_bf16 = False
            self.use_fp16 = True
        os.makedirs(self.output_dir, exist_ok=True)
        os.makedirs(self.plots_dir, exist_ok=True)
        os.makedirs(self.logs_dir, exist_ok=True)
        os.makedirs(self.tensorboard_dir, exist_ok=True)


def torch_bf16_available() -> bool:
    try:
        import torch
        return torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    except Exception:
        return False


### Source: `utils.py`

In [ ]:
import os
import re
import random
import logging
from typing import Optional

import numpy as np
import torch
import pandas as pd


DEVANAGARI_START = 0x0900
DEVANAGARI_END = 0x097F


def detect_language(token: str) -> str:
    if not token or not token.strip():
        return "other"
    token = token.strip()
    if any(DEVANAGARI_START <= ord(c) <= DEVANAGARI_END for c in token):
        return "ne"
    cleaned = re.sub(r'[.,!?;:\'"\-()\[\]{}«»…]', '', token)
    if not cleaned:
        return "other"
    if cleaned.replace('.', '').replace('-', '').isdigit():
        return "num"
    if all(c.isascii() and c.isalpha() for c in cleaned):
        return "en"
    return "other"


def set_seed(seed: int = 43) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def setup_logging(name: Optional[str] = None, level: int = logging.INFO) -> logging.Logger:
    logger = logging.getLogger(name or __name__)
    if not logger.handlers:
        handler = logging.StreamHandler()
        handler.setFormatter(logging.Formatter(
            '%(asctime)s - %(name)s - %(levelname)s - %(message)s'
        ))
        logger.addHandler(handler)
    logger.setLevel(level)
    return logger


logger = setup_logging(__name__)


AUDIO_KEYWORDS = ["file", "path", "audio", "filename", "wav", "wav_path"]
TEXT_KEYWORDS = [
    "text", "transcript", "sentence", "transcription",
    "label", "utterance", "ref", "reference"
]


def detect_columns(df: pd.DataFrame) -> tuple[str, str]:
    audio_col = None
    text_col = None
    for col in df.columns:
        col_lower = col.lower().strip()
        if any(kw in col_lower for kw in AUDIO_KEYWORDS):
            if audio_col is None:
                audio_col = col
        if any(kw in col_lower for kw in TEXT_KEYWORDS):
            if text_col is None:
                text_col = col
    if audio_col is None:
        audio_col = df.columns[0]
        logger.warning(f"Audio column not detected, using first column: {audio_col}")
    if text_col is None:
        if len(df.columns) > 1:
            text_col = df.columns[1]
        else:
            text_col = df.columns[0]
        logger.warning(f"Text column not detected, using: {text_col}")
    logger.info(f"Detected columns — audio: '{audio_col}', text: '{text_col}'")
    return audio_col, text_col


### Source: `metrics.py`

In [ ]:
import re
from collections import Counter, defaultdict
from typing import Dict, List, Optional, Tuple

import jiwer
import numpy as np
import pandas as pd
from tqdm import tqdm



logger = setup_logging(__name__)


def compute_wer(reference: str, hypothesis: str) -> float:
    if not reference and not hypothesis:
        return 0.0
    if not reference:
        return 1.0
    if not hypothesis:
        return 1.0
    return jiwer.wer(reference, hypothesis)


def compute_cer(reference: str, hypothesis: str) -> float:
    if not reference and not hypothesis:
        return 0.0
    if not reference:
        return 1.0
    if not hypothesis:
        return 1.0
    return jiwer.cer(reference, hypothesis)


def filter_text_by_language(text: str, lang: str) -> str:
    tokens = text.split()
    filtered = [t for t in tokens if detect_language(t) == lang]
    return " ".join(filtered)


def compute_wer_nepali(reference: str, hypothesis: str) -> float:
    ref_ne = filter_text_by_language(reference, "ne")
    hyp_ne = filter_text_by_language(hypothesis, "ne")
    if not ref_ne and not hyp_ne:
        return 0.0
    if not ref_ne:
        return float('inf')
    return jiwer.wer(ref_ne, hyp_ne)


def compute_wer_english(reference: str, hypothesis: str) -> float:
    ref_en = filter_text_by_language(reference, "en")
    hyp_en = filter_text_by_language(hypothesis, "en")
    if not ref_en and not hyp_en:
        return 0.0
    if not ref_en:
        return float('inf')
    return jiwer.wer(ref_en, hyp_en)


def compute_mer(
    references: List[str],
    hypotheses: List[str],
) -> float:
    total_s = total_i = total_d = total_n = 0
    for ref, hyp in zip(references, hypotheses):
        if not ref and not hyp:
            continue
        out = jiwer.process_words(ref, hyp)
        total_s += out.substitutions
        total_i += out.insertions
        total_d += out.deletions
        total_n += len(out.references[0]) if out.references else 0
    denom = total_n + total_i + total_d
    if denom == 0:
        return 0.0
    return (total_s + total_i + total_d) / denom


def compute_mer_per_language(
    references: List[str],
    hypotheses: List[str],
) -> Dict[str, float]:
    counts: Dict[str, Dict[str, int]] = {
        lang: {"s": 0, "i": 0, "d": 0, "n": 0}
        for lang in ("ne", "en")
    }
    for ref, hyp in zip(references, hypotheses):
        if not ref and not hyp:
            continue
        try:
            out = jiwer.process_words(ref, hyp)
        except Exception:
            continue

        ref_tokens = out.references[0] if out.references else []
        hyp_tokens = out.hypotheses[0] if out.hypotheses else []
        alignments = out.alignments[0] if out.alignments else []

        for chunk in alignments:
            lang = "other"
            if chunk.type == "equal":
                for idx in range(chunk.ref_start_idx, chunk.ref_end_idx):
                    lang = detect_language(ref_tokens[idx]) if idx < len(ref_tokens) else "other"
                    if lang in counts:
                        counts[lang]["n"] += 1
            elif chunk.type == "substitute":
                for idx in range(chunk.ref_start_idx, chunk.ref_end_idx):
                    lang = detect_language(ref_tokens[idx]) if idx < len(ref_tokens) else "other"
                    if lang in counts:
                        counts[lang]["s"] += 1
                        counts[lang]["n"] += 1
            elif chunk.type == "delete":
                for idx in range(chunk.ref_start_idx, chunk.ref_end_idx):
                    lang = detect_language(ref_tokens[idx]) if idx < len(ref_tokens) else "other"
                    if lang in counts:
                        counts[lang]["d"] += 1
                        counts[lang]["n"] += 1
            elif chunk.type == "insert":
                for idx in range(chunk.hyp_start_idx, chunk.hyp_end_idx):
                    lang = detect_language(hyp_tokens[idx]) if idx < len(hyp_tokens) else "other"
                    if lang in counts:
                        counts[lang]["i"] += 1

    results = {}
    for lang in ("ne", "en"):
        c = counts[lang]
        denom = c["n"] + c["i"] + c["d"]
        if denom == 0:
            results[f"MER_{lang}"] = 0.0
        else:
            results[f"MER_{lang}"] = (c["s"] + c["i"] + c["d"]) / denom
    return results


def compute_token_statistics(
    references: List[str],
    hypotheses: List[str],
) -> Dict:
    nepali_words = 0
    english_words = 0
    mixed_sentences = 0
    total_sentences = len(references)
    total_tokens = 0
    vocab = Counter()
    lang_switches = 0

    for ref in references:
        if not ref:
            continue
        tokens = ref.split()
        total_tokens += len(tokens)

        langs = [detect_language(t) for t in tokens]
        has_ne = "ne" in langs
        has_en = "en" in langs
        has_other = any(l not in ("ne", "en") for l in langs)

        nepali_words += sum(1 for l in langs if l == "ne")
        english_words += sum(1 for l in langs if l == "en")

        if has_ne and (has_en or has_other):
            mixed_sentences += 1

        for t in tokens:
            vocab[t.lower()] += 1

        for i in range(1, len(langs)):
            if langs[i] != langs[i - 1]:
                lang_switches += 1

    avg_len = total_tokens / total_sentences if total_sentences > 0 else 0
    smaller = min(nepali_words, english_words)
    total_ne_en = nepali_words + english_words
    cm_ratio = 1.0 - (smaller / total_ne_en) if total_ne_en > 0 else 0.0

    switch_freq = lang_switches / total_tokens if total_tokens > 0 else 0

    return {
        "total_sentences": total_sentences,
        "total_tokens": total_tokens,
        "num_nepali_words": nepali_words,
        "num_english_words": english_words,
        "num_mixed_sentences": mixed_sentences,
        "avg_sentence_length": avg_len,
        "vocabulary_size": len(vocab),
        "code_mixing_ratio": cm_ratio,
        "language_switching_frequency": switch_freq,
    }


def analyze_errors(
    references: List[str],
    hypotheses: List[str],
    top_k: int = 50,
) -> Dict:
    sub_counter = Counter()
    del_counter = Counter()
    ins_counter = Counter()
    nepali_errors = Counter()
    english_errors = Counter()
    sentence_wer = []
    sentence_cer = []
    conf_matrix: Dict[str, Dict[str, int]] = defaultdict(lambda: defaultdict(int))

    for ref, hyp in tqdm(
        zip(references, hypotheses),
        total=len(references),
        desc="Analyzing errors",
    ):
        if not ref and not hyp:
            sentence_wer.append(0.0)
            sentence_cer.append(0.0)
            continue

        sentence_wer.append(jiwer.wer(ref, hyp))
        sentence_cer.append(jiwer.cer(ref, hyp))

        try:
            out = jiwer.process_words(ref, hyp)
        except Exception:
            continue

        ref_tokens = out.references[0] if out.references else []
        hyp_tokens = out.hypotheses[0] if out.hypotheses else []
        alignments = out.alignments[0] if out.alignments else []

        for chunk in alignments:
            if chunk.type == "equal":
                for idx in range(chunk.ref_start_idx, chunk.ref_end_idx):
                    if idx < len(ref_tokens):
                        lang = detect_language(ref_tokens[idx])
                        conf_matrix[lang]["correct"] += 1

            elif chunk.type == "substitute":
                for idx in range(chunk.ref_start_idx, chunk.ref_end_idx):
                    if idx < len(ref_tokens):
                        word = ref_tokens[idx]
                        pair = (word, hyp_tokens[idx] if idx < len(hyp_tokens) else "")
                        sub_counter[pair] += 1
                        lang = detect_language(word)
                        conf_matrix[lang]["substituted"] += 1
                        if lang == "ne":
                            nepali_errors[word] += 1
                        elif lang == "en":
                            english_errors[word] += 1

            elif chunk.type == "delete":
                for idx in range(chunk.ref_start_idx, chunk.ref_end_idx):
                    if idx < len(ref_tokens):
                        word = ref_tokens[idx]
                        del_counter[word] += 1
                        lang = detect_language(word)
                        conf_matrix[lang]["deleted"] += 1
                        if lang == "ne":
                            nepali_errors[word] += 1
                        elif lang == "en":
                            english_errors[word] += 1

            elif chunk.type == "insert":
                for idx in range(chunk.hyp_start_idx, chunk.hyp_end_idx):
                    if idx < len(hyp_tokens):
                        word = hyp_tokens[idx]
                        ins_counter[word] += 1
                        lang = detect_language(word)
                        conf_matrix[lang]["inserted"] += 1

    worst_indices = np.argsort(sentence_wer)[-20:][::-1]
    best_indices = np.argsort(sentence_wer)[:20]

    return {
        "substitutions": sub_counter.most_common(top_k),
        "deletions": del_counter.most_common(top_k),
        "insertions": ins_counter.most_common(top_k),
        "nepali_errors": nepali_errors.most_common(top_k),
        "english_errors": english_errors.most_common(top_k),
        "sentence_wer": sentence_wer,
        "sentence_cer": sentence_cer,
        "confusion_matrix": {k: dict(v) for k, v in conf_matrix.items()},
        "worst_predictions": [
            {"index": int(i), "wer": float(sentence_wer[i]), "reference": references[i],
             "hypothesis": hypotheses[i]}
            for i in worst_indices if i < len(references)
        ],
        "best_predictions": [
            {"index": int(i), "wer": float(sentence_wer[i]), "reference": references[i],
             "hypothesis": hypotheses[i]}
            for i in best_indices if i < len(references)
        ],
    }


### Source: `plots.py`

In [ ]:
import os
from typing import Dict, List, Optional

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from collections import Counter



logger = setup_logging(__name__)

plt.rcParams.update({
    "figure.dpi": 150,
    "figure.figsize": (10, 6),
    "font.size": 11,
})


def plot_training_history(
    history: Dict[str, List[float]],
    save_dir: str,
) -> None:
    os.makedirs(save_dir, exist_ok=True)

    train_loss = history.get("train_loss", [])
    val_loss = history.get("eval_loss", [])
    wer = history.get("eval_wer", [])
    cer = history.get("eval_cer", [])
    nepali_wer = history.get("eval_wer_nepali", [])
    english_wer = history.get("eval_wer_english", [])
    mer = history.get("eval_mer", [])
    lr = history.get("learning_rate", [])

    if train_loss:
        fig, ax = plt.subplots()
        ax.plot(train_loss, label="Training Loss", color="#2563eb")
        if val_loss:
            ax.plot(val_loss, label="Validation Loss", color="#dc2626")
        ax.set_xlabel("Step")
        ax.set_ylabel("Loss")
        ax.set_title("Loss Curves")
        ax.legend()
        ax.grid(True, alpha=0.3)
        fig.tight_layout()
        fig.savefig(os.path.join(save_dir, "loss_curve.png"))
        plt.close(fig)

    if wer:
        fig, ax = plt.subplots()
        ax.plot(wer, label="WER", color="#2563eb", marker="o")
        if cer:
            ax.plot(cer, label="CER", color="#dc2626", marker="s")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Rate")
        ax.set_title("WER / CER")
        ax.legend()
        ax.grid(True, alpha=0.3)
        fig.tight_layout()
        fig.savefig(os.path.join(save_dir, "wer_cer_curve.png"))
        plt.close(fig)

    if nepali_wer:
        fig, ax = plt.subplots()
        if nepali_wer:
            ax.plot(nepali_wer, label="Nepali WER", color="#2563eb", marker="o")
        if english_wer:
            ax.plot(english_wer, label="English WER", color="#dc2626", marker="s")
        if mer:
            ax.plot(mer, label="MER", color="#16a34a", marker="^")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Rate")
        ax.set_title("Language-aware Metrics")
        ax.legend()
        ax.grid(True, alpha=0.3)
        fig.tight_layout()
        fig.savefig(os.path.join(save_dir, "language_metrics.png"))
        plt.close(fig)

    if lr:
        fig, ax = plt.subplots()
        ax.plot(lr, label="Learning Rate", color="#2563eb")
        ax.set_xlabel("Step")
        ax.set_ylabel("Learning Rate")
        ax.set_title("Learning Rate Schedule")
        ax.legend()
        ax.grid(True, alpha=0.3)
        fig.tight_layout()
        fig.savefig(os.path.join(save_dir, "learning_rate.png"))
        plt.close(fig)

    logger.info(f"Training history plots saved to {save_dir}")


def plot_error_histograms(
    sentence_wer: List[float],
    sentence_cer: List[float],
    save_dir: str,
) -> None:
    os.makedirs(save_dir, exist_ok=True)
    bins = 50

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].hist(sentence_wer, bins=bins, color="#2563eb", alpha=0.7, edgecolor="white")
    axes[0].set_xlabel("WER")
    axes[0].set_ylabel("Count")
    axes[0].set_title("Sentence-level WER Distribution")
    axes[0].axvline(np.mean(sentence_wer), color="red", linestyle="--",
                    label=f"Mean: {np.mean(sentence_wer):.3f}")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].hist(sentence_cer, bins=bins, color="#dc2626", alpha=0.7, edgecolor="white")
    axes[1].set_xlabel("CER")
    axes[1].set_ylabel("Count")
    axes[1].set_title("Sentence-level CER Distribution")
    axes[1].axvline(np.mean(sentence_cer), color="blue", linestyle="--",
                    label=f"Mean: {np.mean(sentence_cer):.3f}")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    fig.tight_layout()
    fig.savefig(os.path.join(save_dir, "error_histograms.png"))
    plt.close(fig)
    logger.info(f"Error histograms saved to {save_dir}")


def plot_codemixing_distribution(
    references: List[str],
    save_dir: str,
) -> None:
    os.makedirs(save_dir, exist_ok=True)

    ratios = []
    for ref in references:
        if not ref:
            continue
        tokens = ref.split()
        if len(tokens) < 2:
            continue
        langs = [detect_language(t) for t in tokens]
        ne_count = sum(1 for l in langs if l == "ne")
        en_count = sum(1 for l in langs if l == "en")
        total = ne_count + en_count
        if total == 0:
            continue
        smaller = min(ne_count, en_count)
        ratio = 1.0 - (smaller / total)
        ratios.append(ratio)

    fig, ax = plt.subplots()
    if ratios:
        ax.hist(ratios, bins=30, color="#16a34a", alpha=0.7, edgecolor="white")
        ax.axvline(np.mean(ratios), color="red", linestyle="--",
                   label=f"Mean: {np.mean(ratios):.3f}")
    ax.set_xlabel("Code-mixing Ratio")
    ax.set_ylabel("Count")
    ax.set_title("Code-mixing Ratio Distribution")
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(save_dir, "codemixing_ratio_distribution.png"))
    plt.close(fig)
    logger.info(f"Code-mixing ratio distribution saved to {save_dir}")


def plot_word_frequency(
    references: List[str],
    save_dir: str,
    top_n: int = 30,
) -> None:
    os.makedirs(save_dir, exist_ok=True)
    word_counts = Counter()
    for ref in references:
        if not ref:
            continue
        word_counts.update(ref.lower().split())

    most_common = word_counts.most_common(top_n)
    words, counts = zip(*most_common) if most_common else ([], [])

    fig, ax = plt.subplots(figsize=(12, 8))
    ax.barh(range(len(words)), counts, color="#8b5cf6", alpha=0.7)
    ax.set_yticks(range(len(words)))
    ax.set_yticklabels(words)
    ax.set_xlabel("Frequency")
    ax.set_title(f"Top {top_n} Most Frequent Words")
    ax.invert_yaxis()
    ax.grid(True, alpha=0.3, axis="x")
    fig.tight_layout()
    fig.savefig(os.path.join(save_dir, "word_frequency.png"))
    plt.close(fig)
    logger.info(f"Word frequency plot saved to {save_dir}")


def generate_all_plots(
    history: Dict[str, List[float]],
    references: List[str],
    sentence_wer: List[float],
    sentence_cer: List[float],
    save_dir: str,
) -> None:
    logger.info("Generating all plots...")
    os.makedirs(save_dir, exist_ok=True)
    plot_training_history(history, save_dir)
    plot_error_histograms(sentence_wer, sentence_cer, save_dir)
    plot_codemixing_distribution(references, save_dir)
    plot_word_frequency(references, save_dir)
    logger.info(f"All plots saved to {save_dir}")


### Source: `dataset.py`

In [ ]:
import os
import csv
import unicodedata

import pandas as pd
import numpy as np
from datasets import Dataset, DatasetDict, Audio, Features, Value
from transformers import WhisperProcessor
from tqdm import tqdm



logger = setup_logging(__name__)


def read_csv_robust(csv_path: str) -> pd.DataFrame:
    paths = []
    texts = []
    with open(csv_path, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        header = next(reader)
        for row in reader:
            if not row:
                continue
            paths.append(row[0].strip())
            texts.append(",".join(row[1:]).strip())
    header_names = [header[0].strip(), header[1].strip()]
    return pd.DataFrame({header_names[0]: paths, header_names[1]: texts})


def clean_dataframe(
    df: pd.DataFrame,
    audio_col: str,
    text_col: str,
    audio_dir: str,
) -> pd.DataFrame:
    logger.info(f"Initial samples: {len(df)}")

    df = df.dropna(subset=[audio_col, text_col])
    logger.info(f"After dropping NaN: {len(df)}")

    df[text_col] = df[text_col].astype(str).str.strip()
    df[text_col] = df[text_col].apply(lambda x: unicodedata.normalize("NFC", x))

    df = df[df[text_col] != ""]
    logger.info(f"After removing empty transcripts: {len(df)}")

    original_audio_col = audio_col
    audio_path_col = "audio_path"
    def resolve_audio_path(row):
        filename = str(row[original_audio_col]).strip()
        if os.path.isabs(filename):
            return filename
        return os.path.join(audio_dir, os.path.basename(filename))

    df[audio_path_col] = df.apply(resolve_audio_path, axis=1)

    existing = df[audio_path_col].apply(os.path.exists)
    logger.info(f"Existing audio files: {existing.sum()} / {len(df)}")
    df = df[existing].reset_index(drop=True)

    for fpath in tqdm(df[audio_path_col], desc="Checking audio files"):
        try:
            import soundfile as sf
            sf.info(fpath)
        except Exception as e:
            logger.warning(f"Corrupted audio: {fpath} — {e}")
            df = df[df[audio_path_col] != fpath]

    logger.info(f"Final samples: {len(df)}")
    return df


def train_validation_split(
    df: pd.DataFrame,
    train_ratio: float = 0.9,
    seed: int = 42,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    set_seed(seed)
    shuffled = df.sample(frac=1, random_state=seed).reset_index(drop=True)
    split_idx = int(len(shuffled) * train_ratio)
    train_df = shuffled.iloc[:split_idx].reset_index(drop=True)
    val_df = shuffled.iloc[split_idx:].reset_index(drop=True)
    logger.info(f"Train: {len(train_df)}, Validation: {len(val_df)}")
    return train_df, val_df


def load_and_prepare_dataset(
    csv_path: str,
    audio_dir: str,
    processor: WhisperProcessor,
    target_sampling_rate: int = 16000,
    train_ratio: float = 0.9,
    seed: int = 42,
    max_audio_length_sec: float = 30.0,
) -> DatasetDict:
    df = read_csv_robust(csv_path)
    audio_col, text_col = detect_columns(df)
    df = clean_dataframe(df, audio_col, text_col, audio_dir)
    train_df, val_df = train_validation_split(df, train_ratio, seed)

    train_dataset = Dataset.from_pandas(train_df)
    val_dataset = Dataset.from_pandas(val_df)

    max_audio_samples = int(target_sampling_rate * max_audio_length_sec)

    def prepare(batch):
        path = batch["audio_path"]
        try:
            import librosa
            array, sr = librosa.load(path, sr=target_sampling_rate, mono=True)
        except Exception:
            import soundfile as sf
            array, sr = sf.read(path)
            if len(array.shape) > 1:
                array = array.mean(axis=1)
            if sr != target_sampling_rate:
                import librosa
                array = librosa.resample(array, orig_sr=sr, target_sr=target_sampling_rate)
                sr = target_sampling_rate

        if len(array) > max_audio_samples:
            array = array[:max_audio_samples]

        batch["input_features"] = processor.feature_extractor(
            array, sampling_rate=sr
        ).input_features[0]

        batch["labels"] = processor.tokenizer(
            batch[text_col],
            truncation=True,
            max_length=448,  # Whisper decoder max_target_positions
        ).input_ids

        batch["duration"] = len(array) / sr
        return batch

    columns_to_remove = [c for c in train_dataset.column_names
                         if c not in ("input_features", "labels", "duration")]

    logger.info("Processing training dataset...")
    train_dataset = train_dataset.map(
        prepare,
        remove_columns=columns_to_remove,
        desc="Processing train",
    )

    logger.info("Processing validation dataset...")
    val_dataset = val_dataset.map(
        prepare,
        remove_columns=columns_to_remove,
        desc="Processing val",
    )

    return DatasetDict({"train": train_dataset, "validation": val_dataset})


### Source: `data_collator.py`

In [ ]:
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Union

import torch
from transformers import WhisperProcessor


@dataclass
class SpeechSeq2SeqDataCollator:
    processor: WhisperProcessor
    model_dtype: torch.dtype = torch.float32
    padding: Union[bool, str] = "longest"

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        input_features = [
            {"input_features": f["input_features"]} for f in features
        ]
        label_features = [
            {"input_ids": f["labels"]} for f in features
        ]

        batch = self.processor.feature_extractor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            padding=self.padding,
            return_tensors="pt",
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch["attention_mask"].ne(1), -100
        )

        batch["labels"] = labels
        batch["input_features"] = batch["input_features"].to(self.model_dtype)

        if "attention_mask" not in batch:
            batch["attention_mask"] = (
                batch["input_features"].abs().sum(dim=-1) != 0
            ).float()

        return batch


### Source: `train.py` (CONTINUATION version)

**Critical differences from original for safe resumption:**
1. Copies the latest checkpoint from previous run's output into working dir
2. Uses lower LR (3e-4) + linear scheduler to avoid forgetting
3. Zero warmup since model is already trained
4. Restores optimizer state, scheduler state, and RNG state from checkpoint

In [ ]:
import os
import sys
import json
import logging
from dataclasses import asdict
from typing import Dict, List, Optional

import jiwer
import numpy as np
import torch
import transformers
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback,
    TrainerCallback,
    TrainingArguments,
    TrainerState,
    TrainerControl,
)
from transformers.trainer_utils import EvalPrediction
from accelerate import Accelerator
from datasets import DatasetDict







logger = setup_logging(__name__)


class MetricsCallback(TrainerCallback):
    def __init__(self, config: TrainingConfig):
        self.config = config
        self.train_losses: List[float] = []
        self.eval_metrics: Dict[str, List[float]] = {}

    def on_log(self, args: TrainingArguments, state: TrainerState,
               control: TrainerControl, **kwargs):
        if state.log_history:
            logs = state.log_history[-1]
            if "loss" in logs:
                self.train_losses.append(logs["loss"])
        return control

    def on_evaluate(self, args: TrainingArguments, state: TrainerState,
                    control: TrainerControl, **kwargs):
        if state.log_history:
            for logs in state.log_history:
                if "eval_loss" in logs:
                    for key, val in logs.items():
                        if key.startswith("eval_") or key == "epoch":
                            if key not in self.eval_metrics:
                                self.eval_metrics[key] = []
                            self.eval_metrics[key].append(val)
        return control



import time
class TimeStoppingCallback(TrainerCallback):
    def __init__(self, max_time_hours=10.5):
        self.start_time = time.time()
        self.max_time_sec = max_time_hours * 3600
        
    def on_step_end(self, args: TrainingArguments, state: TrainerState, control: TrainerControl, **kwargs):
        elapsed = time.time() - self.start_time
        if elapsed > self.max_time_sec:
            logger.warning(f"Stopping training early because {elapsed/3600:.2f} hours elapsed (Kaggle limit protection).")
            control.should_training_stop = True
            control.should_save = True
        return control

class GPUMemoryCallback(TrainerCallback):
    def on_step_end(self, args: TrainingArguments, state: TrainerState,
                    control: TrainerControl, **kwargs):
        if torch.cuda.is_available() and state.global_step % 50 == 0:
            allocated = torch.cuda.max_memory_allocated() / (1024 ** 3)
            reserved = torch.cuda.max_memory_reserved() / (1024 ** 3)
            logger.info(
                f"Step {state.global_step} — "
                f"GPU memory: {allocated:.2f} GB allocated, "
                f"{reserved:.2f} GB reserved"
            )
        return control


def compute_metrics_fn(
    processor: WhisperProcessor,
    eval_pred: EvalPrediction,
) -> Dict[str, float]:
    pred_ids = eval_pred.predictions
    label_ids = eval_pred.label_ids

    if isinstance(pred_ids, tuple):
        pred_ids = pred_ids[0]

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)

    pred_str = [p.strip() for p in pred_str]
    label_str = [l.strip() for l in label_str]

    total_wer = np.mean([compute_wer(l, p) for l, p in zip(label_str, pred_str)])
    total_cer = np.mean([compute_cer(l, p) for l, p in zip(label_str, pred_str)])

    nepali_wer_values = [
        v for v in [compute_wer_nepali(l, p) for l, p in zip(label_str, pred_str)]
        if v != float('inf')
    ]
    nepali_wer = np.mean(nepali_wer_values) if nepali_wer_values else 0.0

    english_wer_values = [
        v for v in [compute_wer_english(l, p) for l, p in zip(label_str, pred_str)]
        if v != float('inf')
    ]
    english_wer = np.mean(english_wer_values) if english_wer_values else 0.0

    mer = compute_mer(label_str, pred_str)
    mer_lang = compute_mer_per_language(label_str, pred_str)

    return {
        "wer": total_wer,
        "cer": total_cer,
        "wer_nepali": nepali_wer,
        "wer_english": english_wer,
        "mer": mer,
        "mer_nepali": mer_lang.get("MER_ne", 0.0),
        "mer_english": mer_lang.get("MER_en", 0.0),
    }


def main():
    config = TrainingConfig()
    set_seed(config.random_seed)
    os.makedirs(config.output_dir, exist_ok=True)

    logger.info("=" * 60)
    logger.info("Whisper Large-v3 — CONTINUATION Training")
    logger.info("=" * 60)
    logger.info(f"Output directory: {config.output_dir}")
    logger.info(f"Model: {config.model_name}")
    logger.info(f"Dataset: {config.csv_path}")
    logger.info(f"Previous output: {config.prev_output_dir}")
    logger.info(f"Learning rate: {config.learning_rate} (was 1e-3)")
    logger.info(f"LR scheduler: {config.lr_scheduler_type} (was cosine)")
    logger.info(f"Warmup ratio: {config.warmup_ratio} (was 0.05)")

    if torch.cuda.is_available():
        n_gpu = torch.cuda.device_count()
        logger.info(f"Using {n_gpu} GPUs")
        for i in range(n_gpu):
            logger.info(f"  GPU {i}: {torch.cuda.get_device_name(i)} "
                        f"({torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB)")

    logger.info("Loading processor...")
    processor = WhisperProcessor.from_pretrained(
        config.model_name,
        language="ne",
        task="transcribe",
    )

    logger.info("Loading model...")
    model = WhisperForConditionalGeneration.from_pretrained(
        config.model_name,
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
        use_safetensors=True,
    )

    model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(
        language="ne", task="transcribe"
    )
    model.generation_config.suppress_tokens = []

    if config.gradient_checkpointing:
        model.config.use_cache = False
    else:
        model.config.use_cache = True

    if config.freeze_encoder:
        for param in model.model.encoder.parameters():
            param.requires_grad = False
        logger.info("Encoder frozen (decoder remains trainable for LoRA)")

    if config.use_lora:
        from peft import LoraConfig, get_peft_model
        lora_config = LoraConfig(
            r=config.lora_r,
            lora_alpha=config.lora_alpha,
            target_modules=list(config.lora_target_modules),
            lora_dropout=config.lora_dropout,
            bias="none",
        )
        from peft import PeftModel
        if config.prev_output_dir and os.path.exists(os.path.join(config.prev_output_dir, 'adapter_config.json')):
            logger.info(f'Loading existing LoRA adapter from {config.prev_output_dir}')
            model = PeftModel.from_pretrained(model, config.prev_output_dir, is_trainable=True)
        else:
            logger.info('Initializing new LoRA adapter')
            model = get_peft_model(model, lora_config)
        if config.gradient_checkpointing:
            model.enable_input_require_grads()
        logger.info("Applied LoRA adapters:")
        model.print_trainable_parameters()

    if torch.cuda.device_count() <= 1:
        model.to("cuda" if torch.cuda.is_available() else "cpu")
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    logger.info(f"Total params: {total_params / 1e6:.2f}M")
    logger.info(f"Trainable params: {trainable_params / 1e6:.2f}M")

    logger.info("Loading and preparing dataset...")
    dataset = load_and_prepare_dataset(
        csv_path=config.csv_path,
        audio_dir=config.audio_dir,
        processor=processor,
        target_sampling_rate=config.target_sampling_rate,
        train_ratio=config.train_ratio,
        seed=config.random_seed,
        max_audio_length_sec=config.max_audio_length_sec,
    )

    data_collator = SpeechSeq2SeqDataCollator(
        processor=processor,
        model_dtype=model.dtype,
        padding="longest",
    )

    training_args = Seq2SeqTrainingArguments(
        output_dir=config.output_dir,
        per_device_train_batch_size=config.per_device_train_batch_size,
        per_device_eval_batch_size=config.per_device_eval_batch_size,
        learning_rate=config.learning_rate,
        weight_decay=config.weight_decay,
        num_train_epochs=config.num_epochs,
        warmup_steps=0,
        lr_scheduler_type=config.lr_scheduler_type,
        optim=config.optim,
        gradient_checkpointing=config.gradient_checkpointing,
        gradient_accumulation_steps=config.gradient_accumulation_steps,
        fp16=config.use_fp16,
        bf16=config.use_bf16,
        eval_strategy="steps",
        eval_steps=500,
        save_strategy="steps",
        save_steps=200,
        save_total_limit=config.save_total_limit,
        load_best_model_at_end=False,  # Disabled: we save the final model explicitly
        metric_for_best_model=config.metric_for_best_model,
        greater_is_better=config.greater_is_better,
        predict_with_generate=config.predict_with_generate,
        generation_max_length=config.max_generation_length,
        logging_steps=config.logging_steps,
        
        report_to=["tensorboard"] if config.tensorboard_logging else [],
        remove_unused_columns=False,
        label_names=["labels"],
        dataloader_num_workers=4,
        seed=config.random_seed,
        data_seed=config.random_seed,
        ddp_find_unused_parameters=False if torch.cuda.device_count() > 1 else None,
        group_by_length=True,
        length_column_name="duration",
    )

    metrics_callback = MetricsCallback(config)
    gpu_memory_callback = GPUMemoryCallback()

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=dataset["train"],
        eval_dataset=dataset["validation"],
        data_collator=data_collator,
        # compute_metrics disabled during training (predict_with_generate=False)
        # Full WER/CER evaluation runs in the evaluate() step after training completes
        processing_class=processor.feature_extractor,
        callbacks=[metrics_callback, gpu_memory_callback, TimeStoppingCallback(max_time_hours=10.5)],
    )


    # ══════════════════════════════════════════════════════════════
    logger.info("Starting training from scratch...")
    resume_path = config.prev_output_dir if config.prev_output_dir and os.path.exists(os.path.join(config.prev_output_dir, 'trainer_state.json')) else None
    if resume_path:
        logger.info(f'Resuming training from checkpoint {resume_path} (optimizer states will be restored)')
    else:
        logger.info('Starting training (adapter weights may be loaded, but optimizer state starts fresh)')
    train_result = trainer.train(resume_from_checkpoint=resume_path)
    trainer.save_model(config.best_checkpoint_dir)
    processor.save_pretrained(config.best_checkpoint_dir)

    with open(os.path.join(config.output_dir, "train_results.json"), "w") as f:
        json.dump({
            "global_step": train_result.global_step,
            "training_loss": train_result.training_loss,
            "metrics": train_result.metrics if hasattr(train_result, "metrics") else {},
        }, f, indent=2)

    # ── Merge training history from previous run + this run ──
    history = {}
    history["train_loss"] = metrics_callback.train_losses
    for key, values in metrics_callback.eval_metrics.items():
        history[key] = values
    with open(os.path.join(config.output_dir, "training_history.json"), "w") as f:
        json.dump(history, f, indent=2)



### Source: `evaluate.py`

In [ ]:
import os
import sys
import json
import time
import unicodedata
from typing import Dict, List, Optional, Tuple
from dataclasses import asdict

import jiwer
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
)
from datasets import Dataset, Audio








logger = setup_logging(__name__)


def run_inference(
    model: WhisperForConditionalGeneration,
    processor: WhisperProcessor,
    audio_paths: List[str],
    references: List[str],
    batch_size: int = 8,
    device: Optional[torch.device] = None,
) -> List[str]:
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model.eval()
    predictions = []
    total_inference_time = 0.0
    total_audio_duration = 0.0

    import librosa

    for i in tqdm(range(0, len(audio_paths), batch_size), desc="Running inference"):
        batch_paths = audio_paths[i:i + batch_size]
        batch_refs = references[i:i + batch_size]

        batch_arrays = []
        batch_durations = []
        for path in batch_paths:
            array, sr = librosa.load(path, sr=16000, mono=True)
            batch_arrays.append(array)
            batch_durations.append(len(array) / sr)

        inputs = processor.feature_extractor(
            batch_arrays,
            sampling_rate=16000,
            return_tensors="pt",
            padding=True,
        ).to(device)
        inputs["input_features"] = inputs["input_features"].to(model.dtype)

        start_time = time.time()
        with torch.no_grad():
            generated_ids = model.generate(
                input_features=inputs["input_features"],
                language="ne",
                task="transcribe",
                max_length=225,
            )
        batch_inference_time = time.time() - start_time
        total_inference_time += batch_inference_time
        total_audio_duration += sum(batch_durations)

        preds = processor.batch_decode(generated_ids, skip_special_tokens=True)
        predictions.extend(preds)

    avg_rtf = total_inference_time / total_audio_duration if total_audio_duration > 0 else 0
    return predictions, avg_rtf


def evaluate():
    config = TrainingConfig()
    set_seed(config.random_seed)

    logger.info("=" * 60)
    logger.info("Evaluation — Nepali-English Code-Mixed ASR")
    logger.info("=" * 60)

    checkpoint_dir = config.best_checkpoint_dir
    if not os.path.exists(checkpoint_dir):
        logger.warning(f"Best checkpoint not found at {checkpoint_dir}")
        checkpoints = [
            d for d in os.listdir(config.output_dir)
            if d.startswith("checkpoint-") and os.path.isdir(os.path.join(config.output_dir, d))
        ]
        if checkpoints:
            last = sorted(checkpoints, key=lambda x: int(x.split("-")[1]))[-1]
            checkpoint_dir = os.path.join(config.output_dir, last)
            logger.info(f"Using latest checkpoint: {checkpoint_dir}")
        else:
            logger.error("No checkpoint found. Train the model first.")
            sys.exit(1)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    logger.info(f"Loading processor from {checkpoint_dir}")
    try:
        processor = WhisperProcessor.from_pretrained(checkpoint_dir)
    except Exception:
        processor = WhisperProcessor.from_pretrained(config.model_name)

    adapter_config_path = os.path.join(checkpoint_dir, "adapter_config.json")
    if os.path.exists(adapter_config_path):
        from peft import PeftModel
        logger.info(f"Loading base {config.model_name} + LoRA adapter from {checkpoint_dir}")
        base_model = WhisperForConditionalGeneration.from_pretrained(
            config.model_name,
            torch_dtype=torch.bfloat16 if config.use_bf16 else torch.float16,
            low_cpu_mem_usage=True,
        )
        model = PeftModel.from_pretrained(base_model, checkpoint_dir)
        model = model.merge_and_unload().to(device)
    else:
        logger.info(f"Loading model from {checkpoint_dir}")
        model = WhisperForConditionalGeneration.from_pretrained(
            checkpoint_dir,
            torch_dtype=torch.bfloat16 if config.use_bf16 else torch.float16,
            low_cpu_mem_usage=True,
        ).to(device)

    model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(
        language="ne", task="transcribe"
    )

    df = read_csv_robust(config.csv_path)
    audio_col, text_col = detect_columns(df)
    df = clean_dataframe(df, audio_col, text_col, config.audio_dir)

    # Evaluate on the held-out validation split only (same seed as training),
    # not the whole dataset -- keeps final evaluation within the 12h budget.
    _, df = train_validation_split(df, config.train_ratio, config.random_seed)

    audio_paths = df["audio_path"].tolist()
    references = df[text_col].tolist()

    logger.info(f"Running inference on {len(audio_paths)} validation samples...")
    predictions, avg_rtf = run_inference(
        model=model,
        processor=processor,
        audio_paths=audio_paths,
        references=references,
        batch_size=config.per_device_eval_batch_size,
        device=device,
    )

    predictions = [p.strip() for p in predictions]
    references = [r.strip() for r in references]

    logger.info("Computing metrics...")
    sentence_results = []
    total_wer = 0.0
    total_cer = 0.0
    total_wer_ne = 0.0
    total_wer_en = 0.0
    total_mer = 0.0
    ne_count = 0
    en_count = 0

    for ref, hyp in zip(references, predictions):
        wer = compute_wer(ref, hyp)
        cer = compute_cer(ref, hyp)
        wer_ne = compute_wer_nepali(ref, hyp)
        wer_en = compute_wer_english(ref, hyp)

        tokens = ref.split()
        ne_words = sum(1 for t in tokens if detect_language(t) == "ne")
        en_words = sum(1 for t in tokens if detect_language(t) == "en")

        total_wer += wer
        total_cer += cer
        if wer_ne != float('inf'):
            total_wer_ne += wer_ne
            ne_count += 1
        if wer_en != float('inf'):
            total_wer_en += wer_en
            en_count += 1

        sentence_results.append({
            "wer": wer,
            "cer": cer,
            "wer_nepali": wer_ne if wer_ne != float('inf') else None,
            "wer_english": wer_en if wer_en != float('inf') else None,
            "num_nepali_words": ne_words,
            "num_english_words": en_words,
        })

    n = len(references)
    overall_wer = total_wer / n if n > 0 else 0.0
    overall_cer = total_cer / n if n > 0 else 0.0
    overall_wer_ne = total_wer_ne / ne_count if ne_count > 0 else 0.0
    overall_wer_en = total_wer_en / en_count if en_count > 0 else 0.0
    overall_mer = compute_mer(references, predictions)
    mer_lang = compute_mer_per_language(references, predictions)

    stats = compute_token_statistics(references, predictions)
    error_analysis = analyze_errors(references, predictions, top_k=50)

    logger.info(f"Overall WER: {overall_wer:.4f}")
    logger.info(f"Overall CER: {overall_cer:.4f}")
    logger.info(f"Nepali WER: {overall_wer_ne:.4f}")
    logger.info(f"English WER: {overall_wer_en:.4f}")
    logger.info(f"MER: {overall_mer:.4f}")
    logger.info(f"MER Nepali: {mer_lang.get('MER_ne', 0.0):.4f}")
    logger.info(f"MER English: {mer_lang.get('MER_en', 0.0):.4f}")

    logger.info(f"Total samples: {stats['total_sentences']}")
    logger.info(f"Nepali words: {stats['num_nepali_words']}")
    logger.info(f"English words: {stats['num_english_words']}")
    logger.info(f"Code-mixing ratio: {stats['code_mixing_ratio']:.4f}")
    logger.info(f"Vocabulary size: {stats['vocabulary_size']}")

    logger.info("Generating CSV report...")
    csv_data = []
    for i, (ref, hyp) in enumerate(zip(references, predictions)):
        sr = sentence_results[i]
        duration = 0.0
        try:
            import soundfile as sf
            info = sf.info(audio_paths[i])
            duration = info.duration
        except Exception:
            pass
        csv_data.append({
            "audio_file": os.path.basename(audio_paths[i]),
            "reference": ref,
            "prediction": hyp,
            "WER": sr["wer"],
            "CER": sr["cer"],
            "WER_nepali": sr["wer_nepali"] if sr["wer_nepali"] is not None else "",
            "WER_english": sr["wer_english"] if sr["wer_english"] is not None else "",
            "MER": overall_mer,
            "duration": duration,
            "num_nepali_words": sr["num_nepali_words"],
            "num_english_words": sr["num_english_words"],
        })

    csv_path = os.path.join(config.output_dir, "evaluation_results.csv")
    pd.DataFrame(csv_data).to_csv(csv_path, index=False)
    logger.info(f"CSV report saved to {csv_path}")

    logger.info("Generating plots...")
    history_path = os.path.join(config.output_dir, "training_history.json")
    history = {"train_loss": [], "eval_loss": [], "eval_wer": [], "eval_cer": []}
    if os.path.exists(history_path):
        with open(history_path) as f:
            history = json.load(f)

    generate_all_plots(
        history=history,
        references=references,
        sentence_wer=[s["wer"] for s in sentence_results],
        sentence_cer=[s["cer"] for s in sentence_results],
        save_dir=config.plots_dir,
    )

    logger.info("Generating evaluation report...")
    generate_evaluation_report(
        config=config,
        overall_wer=overall_wer,
        overall_cer=overall_cer,
        overall_wer_ne=overall_wer_ne,
        overall_wer_en=overall_wer_en,
        overall_mer=overall_mer,
        mer_lang=mer_lang,
        stats=stats,
        error_analysis=error_analysis,
        references=references,
        predictions=predictions,
        avg_rtf=avg_rtf,
        model=model,
        save_dir=config.output_dir,
    )

    logger.info("Evaluation complete!")
    return {
        "overall_wer": overall_wer,
        "overall_cer": overall_cer,
        "wer_nepali": overall_wer_ne,
        "wer_english": overall_wer_en,
        "mer": overall_mer,
        "mer_nepali": mer_lang.get("MER_ne", 0.0),
        "mer_english": mer_lang.get("MER_en", 0.0),
        "stats": stats,
    }


def generate_evaluation_report(
    config: TrainingConfig,
    overall_wer: float,
    overall_cer: float,
    overall_wer_ne: float,
    overall_wer_en: float,
    overall_mer: float,
    mer_lang: Dict[str, float],
    stats: Dict,
    error_analysis: Dict,
    references: List[str],
    predictions: List[str],
    avg_rtf: float,
    model: torch.nn.Module,
    save_dir: str,
) -> str:
    report_lines = []
    report_lines.append("# Evaluation Report")
    report_lines.append("")
    report_lines.append(f"*Generated on: {time.strftime('%Y-%m-%d %H:%M:%S')}*")
    report_lines.append("")

    report_lines.append("## Training Configuration")
    report_lines.append("")
    report_lines.append(f"| Parameter | Value |")
    report_lines.append(f"|-----------|-------|")
    for key, val in asdict(config).items():
        if not key.startswith("_"):
            report_lines.append(f"| {key} | {val} |")
    report_lines.append("")

    model_size = sum(p.numel() for p in model.parameters())
    report_lines.append(f"| Model Size | {model_size / 1e6:.2f}M params |")

    report_lines.append("")
    report_lines.append("## Dataset Statistics")
    report_lines.append("")
    report_lines.append(f"| Metric | Value |")
    report_lines.append(f"|--------|-------|")
    report_lines.append(f"| Total Samples | {stats['total_sentences']} |")
    report_lines.append(f"| Total Tokens | {stats['total_tokens']} |")
    report_lines.append(f"| Nepali Words | {stats['num_nepali_words']} |")
    report_lines.append(f"| English Words | {stats['num_english_words']} |")
    report_lines.append(f"| Mixed Sentences | {stats['num_mixed_sentences']} |")
    report_lines.append(f"| Avg Sentence Length | {stats['avg_sentence_length']:.2f} |")
    report_lines.append(f"| Vocabulary Size | {stats['vocabulary_size']} |")
    report_lines.append(f"| Code-Mixing Ratio | {stats['code_mixing_ratio']:.4f} |")
    report_lines.append(f"| Language Switching Freq | {stats['language_switching_frequency']:.4f} |")
    report_lines.append("")

    report_lines.append("## Evaluation Metrics")
    report_lines.append("")
    report_lines.append(f"| Metric | Value |")
    report_lines.append(f"|--------|-------|")
    report_lines.append(f"| Overall WER | {overall_wer:.4f} |")
    report_lines.append(f"| Overall CER | {overall_cer:.4f} |")
    report_lines.append(f"| Nepali WER | {overall_wer_ne:.4f} |")
    report_lines.append(f"| English WER | {overall_wer_en:.4f} |")
    report_lines.append(f"| MER | {overall_mer:.4f} |")
    report_lines.append(f"| MER Nepali | {mer_lang.get('MER_ne', 0.0):.4f} |")
    report_lines.append(f"| MER English | {mer_lang.get('MER_en', 0.0):.4f} |")
    report_lines.append(f"| Inference RTF | {avg_rtf:.4f} |")
    report_lines.append("")

    report_lines.append("## Top Substitution Errors")
    report_lines.append("")
    report_lines.append("| Reference | Hypothesis | Count |")
    report_lines.append("|-----------|------------|-------|")
    for (ref_word, hyp_word), count in error_analysis["substitutions"][:20]:
        report_lines.append(f"| {ref_word} | {hyp_word} | {count} |")
    report_lines.append("")

    report_lines.append("## Top Deletions")
    report_lines.append("")
    report_lines.append("| Word | Count |")
    report_lines.append("|------|-------|")
    for word, count in error_analysis["deletions"][:20]:
        report_lines.append(f"| {word} | {count} |")
    report_lines.append("")

    report_lines.append("## Top Insertions")
    report_lines.append("")
    report_lines.append("| Word | Count |")
    report_lines.append("|------|-------|")
    for word, count in error_analysis["insertions"][:20]:
        report_lines.append(f"| {word} | {count} |")
    report_lines.append("")

    report_lines.append("## Top 20 Incorrect Nepali Words")
    report_lines.append("")
    report_lines.append("| Word | Count |")
    report_lines.append("|------|-------|")
    for word, count in error_analysis["nepali_errors"][:20]:
        report_lines.append(f"| {word} | {count} |")
    report_lines.append("")

    report_lines.append("## Top 20 Incorrect English Words")
    report_lines.append("")
    report_lines.append("| Word | Count |")
    report_lines.append("|------|-------|")
    for word, count in error_analysis["english_errors"][:20]:
        report_lines.append(f"| {word} | {count} |")
    report_lines.append("")

    report_lines.append("## Worst 5 Predictions")
    report_lines.append("")
    report_lines.append("| # | Reference | Prediction | WER |")
    report_lines.append("|---|-----------|------------|-----|")
    for i, pred in enumerate(error_analysis["worst_predictions"][:5]):
        ref_short = pred["reference"][:80] + "..." if len(pred["reference"]) > 80 else pred["reference"]
        hyp_short = pred["hypothesis"][:80] + "..." if len(pred["hypothesis"]) > 80 else pred["hypothesis"]
        report_lines.append(f"| {i+1} | {ref_short} | {hyp_short} | {pred['wer']:.4f} |")
    report_lines.append("")

    report_lines.append("## Best 5 Predictions")
    report_lines.append("")
    report_lines.append("| # | Reference | Prediction | WER |")
    report_lines.append("|---|-----------|------------|-----|")
    for i, pred in enumerate(error_analysis["best_predictions"][:5]):
        ref_short = pred["reference"][:80] + "..." if len(pred["reference"]) > 80 else pred["reference"]
        hyp_short = pred["hypothesis"][:80] + "..." if len(pred["hypothesis"]) > 80 else pred["hypothesis"]
        report_lines.append(f"| {i+1} | {ref_short} | {hyp_short} | {pred['wer']:.4f} |")
    report_lines.append("")

    report_lines.append("## Recommendations")
    report_lines.append("")
    report_lines.append("1. **Increase Nepali training data** to improve Nepali WER further.")
    report_lines.append("2. **Add language-adversarial loss** to better handle code-mixed utterances.")
    report_lines.append("3. **Use language-specific tokenizers** for more accurate Nepali character handling.")
    report_lines.append("4. **Data augmentation** with speed perturbation and noise injection for robustness.")
    report_lines.append("5. **Ensemble decoding** with multiple checkpoints for improved accuracy.")
    report_lines.append("6. **Language model fusion** with Nepali-English LM for better contextual predictions.")
    report_lines.append("7. **SpecAugment** for better generalization on limited data.")
    report_lines.append("")

    report_path = os.path.join(save_dir, "evaluation_report.md")
    with open(report_path, "w", encoding="utf-8") as f:
        f.write("\n".join(report_lines))
    logger.info(f"Evaluation report saved to {report_path}")
    return report_path




### Execute Pipeline
Run the continuation training and evaluation.

In [ ]:
print('=' * 60)
print('WHISPER LARGE-V3 LoRA — FROM SCRATCH TRAINING')
print('=' * 60)
print('Starting training from scratch')
print('LR: 1e-3')
print('Scheduler: cosine')
print('Warmup: 0.05')
print('=' * 60)

import traceback

# ── TRAINING ──
print('\n--- Starting Training ---')
try:
    if 'main' in globals():
        trainer, dataset, history = main()
        print(f'\nTraining completed successfully!')
        print(f'Checkpoints saved in: /kaggle/working/outputs/')
except Exception as e:
    print(f'Training Error: {e}')
    traceback.print_exc()

# ── EVALUATION (only if training finished all epochs) ──
# Check if we completed training or were time-stopped
import glob, os
config = TrainingConfig()
checkpoints = glob.glob(os.path.join(config.output_dir, "checkpoint-*"))
best_dir = config.best_checkpoint_dir

if os.path.exists(best_dir):
    print('\n--- Starting Evaluation ---')
    try:
        if 'evaluate' in globals():
            evaluate()
    except Exception as e:
        print(f'Evaluation Error: {e}')
        traceback.print_exc()

    print('\n--- Displaying Plots ---')
    from IPython.display import Image, display
    for plot_name in ['loss_curve.png', 'wer_cer_curve.png', 'language_metrics.png']:
        plot_path = os.path.join(config.plots_dir, plot_name)
        if os.path.exists(plot_path):
            display(Image(filename=plot_path))
else:
    print('\n--- Training was time-stopped (Kaggle 12h limit protection) ---')
    print(f'Saved checkpoints: {[os.path.basename(c) for c in checkpoints]}')
    print('\nTo continue training:')
    print('1. Save this notebook output as a Kaggle dataset')
    print('2. Add that dataset as input to your next run')
    print('3. Re-run this notebook')
    print('\nTo run evaluation on the latest checkpoint:')
    print('Load the checkpoint manually and call evaluate()')
